# FLUX Model Download & Compression

**Objective:** Download Flux-Apex-V1.flx from HuggingFace and compress it for easier distribution.

**What this notebook does:**
1. Downloads Flux-Apex-V1.flx (~17.4GB) from HuggingFace Hub
2. Compresses it using LZMA/gzip for maximum compression
3. Optionally splits into smaller chunks for easier download
4. Provides download links for Kaggle/Colab output

**Expected compression:** ~40-60% reduction (PyTorch files compress well due to tensor redundancy)

In [ ]:
"""Cell 1: Environment Setup"""

import os
import sys
import gc
import gzip
import lzma
import shutil
import zipfile
import tarfile
from pathlib import Path
from datetime import datetime
from typing import Optional

# Environment detection
if Path('/kaggle').exists():
    ROOT = Path('/kaggle/working/FLUX')
    OUTPUT_DIR = Path('/kaggle/working')
    ENVIRONMENT = 'kaggle'
    # Clone repo for utilities
    if not ROOT.exists():
        os.system('git clone https://github.com/Unseengap/FLUX.git /kaggle/working/FLUX')
elif Path('/content').exists():
    ROOT = Path('/content/FLUX')
    OUTPUT_DIR = Path('/content')
    ENVIRONMENT = 'colab'
    if not ROOT.exists():
        os.system('git clone https://github.com/Unseengap/FLUX.git /content/FLUX')
else:
    ROOT = Path('/Users/admin/Desktop/flux')
    OUTPUT_DIR = ROOT / 'checkpoints'
    ENVIRONMENT = 'local'

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print(f"✓ Environment: {ENVIRONMENT}")
print(f"✓ Root: {ROOT}")
print(f"✓ Output directory: {OUTPUT_DIR}")

In [ ]:
"""Cell 2: Setup Utilities & HuggingFace Token"""

from flux_utils import PhaseLogger, _resolve_hf_token

log = PhaseLogger(phase=300)  # Phase 300 = Download/Compress utility
log.separator("FLUX Download & Compression")

# Resolve HuggingFace token
try:
    hf_token = _resolve_hf_token()
    log.success("HuggingFace token resolved")
except:
    hf_token = None
    log.warning("No HF token - may have limited download speed")

# Check disk space
if ENVIRONMENT in ['kaggle', 'colab']:
    import subprocess
    result = subprocess.run(['df', '-h', str(OUTPUT_DIR)], capture_output=True, text=True)
    print("\nDisk space available:")
    print(result.stdout)

In [ ]:
"""Cell 3: Download Flux-Apex-V1.flx from HuggingFace"""

log.cell_start("Cell 3 — Download Model")

from huggingface_hub import hf_hub_download

CHECKPOINT_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'

if CHECKPOINT_PATH.exists():
    log.info(f"Model already exists: {CHECKPOINT_PATH}")
    log.info(f"Size: {CHECKPOINT_PATH.stat().st_size / 1e9:.2f} GB")
else:
    log.info("Downloading Flux-Apex-V1.flx from HuggingFace...")
    log.info("This may take 10-30 minutes depending on connection speed.")
    
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    downloaded = hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(ROOT),
        token=hf_token,
    )
    
    log.success(f"Downloaded: {downloaded}")
    log.info(f"Size: {Path(downloaded).stat().st_size / 1e9:.2f} GB")

original_size = CHECKPOINT_PATH.stat().st_size
log.cell_end("Cell 3 — Download Model", "PASS")

In [ ]:
"""Cell 4: Compression Options"""

# Choose compression method
# Options:
#   'gzip' - Fast compression, decent ratio (~40-50% reduction)
#   'lzma' - Slower, better compression (~50-60% reduction)
#   'zip'  - Compatible everywhere, decent compression
#   'tar.gz' - Unix standard, good compression
#   'tar.xz' - Best compression, slowest

COMPRESSION_METHOD = 'gzip'  # Change this to your preference
CHUNK_SIZE_GB = 4.0  # Split into chunks of this size (0 = no splitting)

print(f"\nCompression settings:")
print(f"  Method: {COMPRESSION_METHOD}")
print(f"  Chunk size: {CHUNK_SIZE_GB}GB" if CHUNK_SIZE_GB > 0 else "  No chunking")
print(f"  Original size: {original_size / 1e9:.2f} GB")

In [ ]:
"""Cell 5: Compress the Model File"""

log.cell_start("Cell 5 — Compression")

import time

def compress_gzip(input_path: Path, output_path: Path) -> Path:
    """Compress using gzip."""
    out = output_path.with_suffix('.flx.gz')
    with open(input_path, 'rb') as f_in:
        with gzip.open(out, 'wb', compresslevel=9) as f_out:
            shutil.copyfileobj(f_in, f_out, length=1024*1024*64)  # 64MB chunks
    return out

def compress_lzma(input_path: Path, output_path: Path) -> Path:
    """Compress using LZMA (xz)."""
    out = output_path.with_suffix('.flx.xz')
    with open(input_path, 'rb') as f_in:
        with lzma.open(out, 'wb', preset=6) as f_out:
            shutil.copyfileobj(f_in, f_out, length=1024*1024*64)
    return out

def compress_zip(input_path: Path, output_path: Path) -> Path:
    """Compress using ZIP."""
    out = output_path.with_suffix('.zip')
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
        zf.write(input_path, input_path.name)
    return out

def compress_tar_gz(input_path: Path, output_path: Path) -> Path:
    """Compress using tar.gz."""
    out = output_path.with_suffix('.tar.gz')
    with tarfile.open(out, 'w:gz', compresslevel=9) as tar:
        tar.add(input_path, arcname=input_path.name)
    return out

def compress_tar_xz(input_path: Path, output_path: Path) -> Path:
    """Compress using tar.xz (best compression)."""
    out = output_path.with_suffix('.tar.xz')
    with tarfile.open(out, 'w:xz', preset=6) as tar:
        tar.add(input_path, arcname=input_path.name)
    return out

# Compression dispatch
compressors = {
    'gzip': compress_gzip,
    'lzma': compress_lzma,
    'zip': compress_zip,
    'tar.gz': compress_tar_gz,
    'tar.xz': compress_tar_xz,
}

output_base = OUTPUT_DIR / 'Flux-Apex-V1'

log.info(f"Compressing with {COMPRESSION_METHOD}...")
log.info("This may take 15-45 minutes depending on the method.")

start_time = time.time()
compressed_path = compressors[COMPRESSION_METHOD](CHECKPOINT_PATH, output_base)
elapsed = time.time() - start_time

compressed_size = compressed_path.stat().st_size
ratio = (1 - compressed_size / original_size) * 100

log.success(f"Compression complete!")
log.info(f"Output: {compressed_path}")
log.info(f"Original: {original_size / 1e9:.2f} GB")
log.info(f"Compressed: {compressed_size / 1e9:.2f} GB")
log.info(f"Reduction: {ratio:.1f}%")
log.info(f"Time: {elapsed/60:.1f} minutes")

log.cell_end("Cell 5 — Compression", "PASS")

In [ ]:
"""Cell 6: Split into Chunks (Optional)"""

log.cell_start("Cell 6 — Chunking")

chunk_paths = []

if CHUNK_SIZE_GB > 0:
    chunk_size_bytes = int(CHUNK_SIZE_GB * 1e9)
    
    if compressed_size > chunk_size_bytes:
        log.info(f"Splitting into {CHUNK_SIZE_GB}GB chunks...")
        
        chunk_dir = OUTPUT_DIR / 'flux_chunks'
        chunk_dir.mkdir(exist_ok=True)
        
        with open(compressed_path, 'rb') as f:
            chunk_num = 0
            while True:
                chunk_data = f.read(chunk_size_bytes)
                if not chunk_data:
                    break
                    
                chunk_file = chunk_dir / f"{compressed_path.stem}.part{chunk_num:02d}"
                with open(chunk_file, 'wb') as cf:
                    cf.write(chunk_data)
                
                chunk_paths.append(chunk_file)
                log.info(f"  Created: {chunk_file.name} ({len(chunk_data)/1e9:.2f} GB)")
                chunk_num += 1
        
        log.success(f"Split into {len(chunk_paths)} chunks")
        
        # Create reassembly script
        reassemble_script = chunk_dir / 'reassemble.sh'
        reassemble_content = f'''#!/bin/bash
# Reassemble Flux-Apex-V1 from chunks
cat {compressed_path.stem}.part* > {compressed_path.name}
echo "Reassembled: {compressed_path.name}"
'''
        reassemble_script.write_text(reassemble_content)
        log.info(f"Created: {reassemble_script}")
    else:
        log.info(f"File is smaller than {CHUNK_SIZE_GB}GB, no chunking needed")
        chunk_paths = [compressed_path]
else:
    log.info("Chunking disabled")
    chunk_paths = [compressed_path]

log.cell_end("Cell 6 — Chunking", "PASS")

In [ ]:
"""Cell 7: Generate Download Summary"""

log.cell_start("Cell 7 — Summary")

print("\n" + "="*60)
print("  FLUX DOWNLOAD & COMPRESSION COMPLETE")
print("="*60)

print(f"\n📦 Original Model:")
print(f"   • Source: HuggingFace UnseenGAP/FLUX")
print(f"   • File: Flux-Apex-V1.flx")
print(f"   • Size: {original_size / 1e9:.2f} GB")

print(f"\n🗜️ Compressed Output:")
print(f"   • File: {compressed_path.name}")
print(f"   • Size: {compressed_size / 1e9:.2f} GB")
print(f"   • Reduction: {ratio:.1f}%")

if len(chunk_paths) > 1:
    print(f"\n📁 Chunks ({len(chunk_paths)} files):")
    for cp in chunk_paths:
        print(f"   • {cp.name}: {cp.stat().st_size / 1e9:.2f} GB")

print(f"\n📂 Output Location: {OUTPUT_DIR}")

# Kaggle/Colab specific instructions
if ENVIRONMENT == 'kaggle':
    print("\n📥 Kaggle Download:")
    print("   Files are in /kaggle/working/")
    print("   Click 'Save Version' → 'Output' to save artifacts")
    print("   Then download from the Output tab")
    
elif ENVIRONMENT == 'colab':
    print("\n📥 Colab Download:")
    print("   Run the next cell to download directly")

log.cell_end("Cell 7 — Summary", "PASS")

In [ ]:
"""Cell 8: Download Files (Colab/Kaggle)"""

# For Colab - trigger browser download
if ENVIRONMENT == 'colab':
    from google.colab import files
    
    print("Initiating download...")
    
    if len(chunk_paths) > 1:
        # Download chunks
        for cp in chunk_paths:
            print(f"Downloading: {cp.name}")
            files.download(str(cp))
    else:
        # Download single file
        files.download(str(compressed_path))
        
    print("\n✓ Download initiated!")

elif ENVIRONMENT == 'kaggle':
    print("Kaggle download instructions:")
    print("1. Click 'Save Version' in the top right")
    print("2. Select 'Save & Run All (Commit)'")
    print("3. After completion, go to your notebook's Output tab")
    print("4. Download the compressed file(s)")
    
    # Copy to output folder for Kaggle
    kaggle_output = Path('/kaggle/working')
    if compressed_path.parent != kaggle_output:
        shutil.copy(compressed_path, kaggle_output / compressed_path.name)
        print(f"\n✓ Copied to: {kaggle_output / compressed_path.name}")

else:
    print(f"\nLocal file ready at:")
    print(f"  {compressed_path}")

In [ ]:
"""Cell 9: Decompression Instructions"""

print("\n" + "="*60)
print("  HOW TO DECOMPRESS")
print("="*60)

decompression_cmds = {
    'gzip': {
        'linux': 'gunzip Flux-Apex-V1.flx.gz',
        'python': '''import gzip, shutil
with gzip.open('Flux-Apex-V1.flx.gz', 'rb') as f_in:
    with open('Flux-Apex-V1.flx', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)'''
    },
    'lzma': {
        'linux': 'xz -d Flux-Apex-V1.flx.xz',
        'python': '''import lzma, shutil
with lzma.open('Flux-Apex-V1.flx.xz', 'rb') as f_in:
    with open('Flux-Apex-V1.flx', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)'''
    },
    'zip': {
        'linux': 'unzip Flux-Apex-V1.zip',
        'python': '''import zipfile
with zipfile.ZipFile('Flux-Apex-V1.zip', 'r') as zf:
    zf.extractall()'''
    },
    'tar.gz': {
        'linux': 'tar -xzf Flux-Apex-V1.tar.gz',
        'python': '''import tarfile
with tarfile.open('Flux-Apex-V1.tar.gz', 'r:gz') as tar:
    tar.extractall()'''
    },
    'tar.xz': {
        'linux': 'tar -xJf Flux-Apex-V1.tar.xz',
        'python': '''import tarfile
with tarfile.open('Flux-Apex-V1.tar.xz', 'r:xz') as tar:
    tar.extractall()'''
    },
}

cmds = decompression_cmds[COMPRESSION_METHOD]

print(f"\n📋 Command Line ({COMPRESSION_METHOD}):")
print(f"   {cmds['linux']}")

print(f"\n🐍 Python:")
print(f"   {cmds['python']}")

if len(chunk_paths) > 1:
    print("\n📦 Reassemble chunks first:")
    print(f"   cat Flux-Apex-V1.*.part* > {compressed_path.name}")

print("\n" + "="*60)

In [ ]:
"""Cell 10: Cleanup (Optional)"""

# Uncomment to delete the original .flx file and free disk space
# WARNING: Only do this if you don't need the uncompressed file!

# import os
# if CHECKPOINT_PATH.exists():
#     os.remove(CHECKPOINT_PATH)
#     print(f"✓ Deleted original: {CHECKPOINT_PATH}")
#     gc.collect()

print("Files retained:")
print(f"  • Original: {CHECKPOINT_PATH} ({original_size/1e9:.2f} GB)")
print(f"  • Compressed: {compressed_path} ({compressed_size/1e9:.2f} GB)")
print("\nUncomment the cleanup code above to remove the original file.")

---

## Usage Notes

### After Downloading:

1. **Decompress** using the commands shown in Cell 9
2. **Load the model**:
   ```python
   from flux_model import FLUXModel
   model = FLUXModel.load('Flux-Apex-V1.flx')
   ```

### Direct HuggingFace Alternative:

If you prefer not to compress, you can download directly via:
```python
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id='UnseenGAP/FLUX',
    filename='checkpoints/Flux-Apex-V1.flx',
    local_dir='.',
)
```

### Model Info:
- **Size**: ~17.4 GB (uncompressed)
- **Parameters**: ~8.34B
- **Format**: PyTorch serialized (.flx)
- **Version**: 8.1-complete